#### 欢迎孟晚舟回家

In [2]:
# 引入必要的库函数
import cv2
import numpy as np

# 读取春笋图像chunsun_origin
chunsun_origin = cv2.imread('chunsun.jpg')
# 读取“孟晚舟欢迎回家”字样图像logo
logo = cv2.imread('logo.jpg')
# 将logo转换为灰度图logo_gray方便二值化获得掩膜
logo_gray = cv2.cvtColor(logo, cv2.COLOR_BGR2GRAY)
# cv2.imshow('logo_gray', logo_gray)
# 获得logo前景foreground
_, mask = cv2.threshold(logo_gray, 200, 255, cv2.THRESH_BINARY_INV)
mask_inv = cv2.bitwise_not(mask)
foreground = cv2.bitwise_and(logo, logo, mask=mask)

# # 获得logo背景chunsun_mask
chunsun_mask = mask_inv

# 可视化foreground和chunsun_mask
cv2.imshow('logo_gray', logo_gray)
cv2.imshow('mask', mask)
cv2.imshow('foreground', foreground)
cv2.imshow('chunsun_mask', chunsun_mask)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [3]:
# 设置保存格式mp4
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# 帧率，即一秒钟播放多少张画面
fps = 30
# 得到摄像头拍摄的视频的宽和高
h, w = chunsun_origin.shape[:2]
# cv2.VideoWriter创建对象，用于视频的写出，文件名保存为welcome.mp4
videoWrite = cv2.VideoWriter('welcome.mp4', fourcc, fps, (w, h))
# 两帧画面之间停顿的时间
delay = int(1000 / fps)


# 控制“孟晚舟欢迎回家”在春笋图片的显示位置x, y = 300, 150
x, y = 300, 150
# 控制“孟晚舟欢迎回家”往上移动的幅度
offset = 2
# 获得logo的h,w
logo_h, logo_w = logo.shape[:2]
# 循环获取摄像头的画面
while True:
    # 从春笋原图拷贝一份图像，待嵌入“孟晚舟欢迎回家”
    chunsun = chunsun_origin.copy()
    # 更新“孟晚舟欢迎回家”在chunsun中的嵌入的位置x, y，（tips:只需更新y值，x保持不变）
    y -= offset 
    # 设置视频结束条件：“孟晚舟欢迎回家”摆放位置的纵坐标y减少到预设的值（例如50），就退出循环
    if y <= 50:
        break
    
    # 把“孟晚舟欢迎回家”logo嵌入到chunsun中。参考上一次图片叠加作业的知识点
    # step1:先选取chunsun的roi区域，
    roi = chunsun[y:y+logo_h, x:x+logo_w]
    # step2:再得到background, background是只有chunsun背景的图片，logo前景部分为黑色
    bg_mask = mask_inv[:roi.shape[0], :roi.shape[1]]
    background = cv2.bitwise_and(roi, roi, mask=bg_mask)
    #step3: 利用cv2.add将background和logo foreground相加
    fg_part = foreground[:roi.shape[0], :roi.shape[1]]
    result = cv2.add(background, fg_part) 
    #step4:  将roi放回原图位置
    chunsun[y:y+logo_h, x:x+logo_w] = result
   
    # 将图片写入视频
    videoWrite.write(chunsun)
    
    # 显示图片
    cv2.imshow('welcome', chunsun)
 
    if (cv2.waitKey(delay) & 0xFF) == ord('q'): 
        break

cv2.waitKey()            
# 刷新，释放资源
videoWrite.release()
cv2.destroyAllWindows()

#### 从下往上滚动

In [4]:

# 读取春笋原图chunsun_origin
chunsun_origin = cv2.imread('chunsun.jpg')
# "孟晚舟欢迎回家"原图logo
logo = cv2.imread('logo.jpg')
 # logo灰度图
logo_gray = cv2.cvtColor(logo, cv2.COLOR_BGR2GRAY)

# 生成logo的掩膜mask
_, mask = cv2.threshold(logo_gray, 200, 255, cv2.THRESH_BINARY_INV)
mask_inv = cv2.bitwise_not(mask)
# logo的前景图像
foreground = cv2.bitwise_and(logo, logo, mask=mask)
# 获得春笋的掩膜
chunsun_bg_mask = mask_inv

# 设置保存格式mp4
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

# 帧率，即一秒钟播放多少张画面
fps = 30

# 得到chunsun宽和高
h, w = chunsun_origin.shape[:2]

# 创建对象，用于视频的写出，文件名命名为welcome2.mp4
videoWrite = cv2.VideoWriter('welcome2.mp4', fourcc, fps, (w, h))

# 两帧画面之间停顿的时间
delay = int(1000 / fps)

#offset 控制“孟晚舟欢迎回家”往上移动的幅度
offset = 3

# 控制“孟晚舟欢迎回家”在春笋图片的显示位置。因为要从下往上滚动，建议y值在原来的位置加上logo的高度
x, y = 300, h - 100

# 设置logo显示多少行，即只显示foreground的前show_y行。初始时logo从底部往上显示，show_y=1
show_y = 1
logo_h, logo_w = logo.shape[:2]

# 循环获取摄像头的画面
while True:
    # 从春笋原图拷贝一份图像，待嵌入“孟晚舟欢迎回家”
    chunsun = chunsun_origin.copy()
    # 如果显示行数show_y还没达到logo_h(完整的logo)，则show_y增加offset
    if show_y < logo_h:
        show_y += offset
    # 防止加上off_set后超过logo_h，show_y取show_y, logo_h两者中的最小值
        show_y = min(show_y, logo_h)
        
    # 设置logo显示部分，取forground的前show_y行    
    current_fg = foreground[:show_y, :]
    # logo在春笋图片中的位置，往上移动offset个单位        
    y -= offset
    
    # 设置视频结束条件
    if y < -show_y:
        break
        
    # 修改背景chunsun图
    # step1:先选取chunsun的roi区域
    roi_y = max(0, y)
    roi_h = min(show_y, h - roi_y)
    if roi_h <= 0: break
    roi = chunsun[roi_y:roi_y+roi_h, x:x+logo_w]
    # step2:再得到background, 此时仅取chunsun_mask的前show_y行作掩膜
    current_mask = mask[:roi_h, :roi.shape[1]]
    current_mask_inv = cv2.bitwise_not(current_mask)
    background = cv2.bitwise_and(roi, roi, mask=current_mask_inv)
    #step3: 利用cv2.add将background和logo foreground相加
    current_fg_crop = foreground[:roi_h, :roi.shape[1]]
    result = cv2.add(background, current_fg_crop)
    #step4:  将roi放回原图位置
    chunsun[roi_y:roi_y+roi_h, x:x+logo_w] = result
    
    # 将图片写入视频
    videoWrite.write(chunsun)
    
    # 显示图片
    cv2.imshow('scrolling', chunsun)
 
    # delay设置不同的值，视频播放速度不一样。
    if (cv2.waitKey(delay) & 0xFF) == 27: 
        break

cv2.waitKey()            
# 刷新，释放资源
videoWrite.release()
cv2.destroyAllWindows()

#### 从下往上滚动+弹幕

In [1]:
import cv2
import numpy as np

# 读取春笋原图chunsun_origin
chunsun_origin = cv2.imread('chunsun.jpg')
# "孟晚舟欢迎回家"原图logo
logo = cv2.imread('logo.jpg')
# logo灰度图
logo_gray = cv2.cvtColor(logo, cv2.COLOR_BGR2GRAY)

# 生成logo的掩膜mask
_, mask = cv2.threshold(logo_gray, 200, 255, cv2.THRESH_BINARY_INV)
mask_inv = cv2.bitwise_not(mask)
# logo的前景图像
foreground = cv2.bitwise_and(logo, logo, mask=mask)
# 获得春笋的掩膜
chunsun_bg_mask = mask_inv

# 设置保存格式mp4
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

# 帧率，即一秒钟播放多少张画面
fps = 30

# 得到chunsun宽和高
h, w = chunsun_origin.shape[:2]

# 创建对象，用于视频的写出，文件名保存为welcomefull.mp4
videoWrite = cv2.VideoWriter('welcome2.mp4', fourcc, fps, (w, h))

# 两帧画面之间停顿的时间
delay = int(1000 / fps)

# offset 控制"孟晚舟欢迎回家"往上移动的幅度
offset = 3

# 控制"孟晚舟欢迎回家"在春笋图片的显示位置。因为要从下往上滚动，建议y值在原来的位置加上logo的高度
x, y = 300, h - 100

# 设置logo显示多少行，即只显示foreground的前show_y行。初始时logo从底部往上显示，show_y=1
show_y = 1
logo_h, logo_w = logo.shape[:2]

# 设置弹幕文本text，如'I love China!'
danmu_text = 'I love China!' 
# 设置弹幕文本text的起始坐标
text_x, text_y = w, 120

# 循环获取摄像头的画面
while True:
    # 从春笋原图拷贝一份图像，待嵌入"孟晚舟欢迎回家"
    chunsun = chunsun_origin.copy()   
    
    # 如果显示行数show_y还没达到logo_h(完整的logo)，则show_y增加offset
    if show_y < logo_h:
        show_y += offset
        # 防止加上off_set后超过logo_h，show_y取show_y, logo_h两者中的最小值
        show_y = min(show_y, logo_h)
        
    # 设置logo显示部分，取forground的前show_y行    
    current_fg = foreground[:show_y, :]  
    
    # 设置视频结束条件
    if y + show_y <= 0:
        break
        
    # 修改背景chunsun图
    # step1:先选取chunsun的roi区域
    roi_y = max(0, y)
    roi_h = min(show_y, h - roi_y)
    if roi_h <= 0: 
        break
    roi = chunsun[roi_y:roi_y+roi_h, x:x+logo_w]
    
    # step2:再得到background, 此时仅取chunsun_mask的前show_y行作掩膜
    current_mask_inv = cv2.bitwise_not(mask[:roi_h, :roi.shape[1]])
    background = cv2.bitwise_and(roi, roi, mask=current_mask_inv)
    
    # step3: 利用cv2.add将background和logo foreground相加
    current_fg_crop = current_fg[:roi_h, :roi.shape[1]]
    result = cv2.add(background, current_fg_crop)
    
    # step4: 将roi放回原图位置 
    chunsun[roi_y:roi_y+roi_h, x:x+logo_w] = result
    
    # 让弹幕向左移动
    text_x -= 5
    # 如果弹幕完全移出屏幕左边，让它从右边重新出现
    if text_x < -len(danmu_text) * 15:
        text_x = w
    
    # cv2.putText设置弹幕
    cv2.putText(chunsun, danmu_text, (text_x, text_y), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2, cv2.LINE_AA)
    
    # 将图片写入视频
    videoWrite.write(chunsun)
    
    # 显示图片
    cv2.imshow('scrolling_danmu', chunsun)
    
    # 【新增】让logo向上移动
    y -= offset
 
    # delay设置不同的值，视频播放速度不一样。
    if (cv2.waitKey(delay) & 0xFF) == 27: 
        break

cv2.waitKey()            
# 刷新，释放资源
videoWrite.release()
cv2.destroyAllWindows()